In [1]:

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
from pathlib import Path


In [2]:

project_path = Path.cwd()

files = {
    "database": "PerfectPetClub.db",
    "crm": "PerfectPetClub_CRM_report.csv",
    "server": "PerfectPetClub_server_log.csv",
    "survey": "PerfectPetClub_survey.csv",
    "questions": "PerfectPetClub_survey_questions.txt"
}

for name, filename in files.items():
    path = project_path / filename
    print(name, filename, path.exists())


database PerfectPetClub.db True
crm PerfectPetClub_CRM_report.csv True
server PerfectPetClub_server_log.csv True
survey PerfectPetClub_survey.csv True
questions PerfectPetClub_survey_questions.txt True


In [3]:

database_path = project_path / files["database"]

connection = sqlite3.connect(database_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    connection
)

display(tables)


,name
0,customer
1,payment
2,product
3,product_category
4,product_review
5,sale
6,sale_product
7,sqlite_sequence
8,subscription
9,subscription_type


In [4]:

table_names = tables["name"].tolist()

for table_name in table_names:
    print(f"\nTable: {table_name}")
    schema = pd.read_sql_query(
        f'PRAGMA table_info("{table_name}")',
        connection
    )
    display(schema[["name", "type"]])



Table: customer


,name,type
0,id,INTEGER
1,name,TEXT
2,email,TEXT
3,password_hash,TEXT
4,address_line_1,TEXT
5,address_line_2,TEXT
6,county,TEXT
7,postcode,TEXT
8,subscription_id,INTEGER



Table: payment


,name,type
0,id,INTEGER
1,amount,INTEGER
2,payment_gateway,TEXT
3,transaction_reference,TEXT



Table: product


,name,type
0,id,INTEGER
1,name,TEXT
2,price,INTEGER
3,product_category_id,INTEGER


... (10 outputs left out)


Table: subscription


,name,type
0,id,INTEGER
1,start_date,TEXT
2,end_date,TEXT
3,auto_renew,INTEGER
4,subscription_type_id,INTEGER



Table: subscription_type


,name,type
0,id,INTEGER
1,name,TEXT
2,delivery_service,TEXT


In [5]:

crm_df = pd.read_csv(project_path / files["crm"])
server_df = pd.read_csv(project_path / files["server"])
survey_df = pd.read_csv(project_path / files["survey"])

print("CRM:", crm_df.shape)
print("Server logs:", server_df.shape)
print("Survey:", survey_df.shape)


CRM: (7585, 4)
Server logs: (67249, 5)
Survey: (221, 5)


In [6]:

datasets = {
    "CRM": crm_df,
    "Server Logs": server_df,
    "Survey": survey_df
}

for name, dataframe in datasets.items():
    print(f"\n{name}")
    print("Columns:", dataframe.columns.tolist())
    display(dataframe.head())



CRM
Columns: ['customer_id', 'recieved_datatime', 'platform', 'type']


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query
3,33,2026-01-01 05:42:16.002,['e-mail'],complaint
4,33,2026-01-01 06:25:03.002,['e-mail'],complaint



Server Logs
Columns: ['datetime', 'session_id', 'ip', 'login', 'user_agent']


,datetime,session_id,ip,login,user_agent
0,2026-01-01 02:08:21.000,session_cookie::63fec84dedaa4f2cbaa10b413b5a7d...,239.66.231.182,Sipho928@somemail.co.uk,"""Mozilla/5.0 (Desktop/Laptop; Cipher CoreBook ..."
1,2026-01-01 02:51:08.001,session_cookie::64bcd32ed1494bd1af6b7a233cd638...,200.200.140.49,Sienna.Martin923@notrealemail.com,"""Mozilla/5.0 (Mobile; Pulse PulsePhone) AppleW..."
2,2026-01-01 04:16:42.001,session_cookie::f848746c9c96422b9eb050eccacfeb...,38.219.218.103,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe..."
3,2026-01-01 04:59:29.002,session_cookie::4589deeb679c4ae68f62a1f61af2db...,126.234.10.95,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe..."
4,2026-01-01 07:07:50.003,session_cookie::2bc1377dd7a74949be4e0035137f5f...,17.99.243.42,io415@fakemail.co.uk,"""Mozilla/5.0 (Tablet; Lumina CorePad) AppleWeb..."



Survey
Columns: ['email', 'reported_income', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq']


,email,reported_income,nps_recommend,satisfaction_score,online_comparison_freq
0,Sargsyan701@somemail.co.uk,26k-30k,8,6,5
1,Mammadova282@example.com,34k-38k,5,8,9
2,Sana858@notrealemail.com,26k-30k,9,5,5
3,Omar.Farooq4@example.com,198k-200k+,9,3,5
4,Alejandro357@somemail.co.uk,42k-46k,0,8,10


In [7]:

for name, dataframe in datasets.items():
    print(f"\n{name}")
    print("Rows:", len(dataframe))
    print("Duplicate rows:", dataframe.duplicated().sum())
    display(dataframe.isna().sum().to_frame("Missing Values"))



CRM
Rows: 7585
Duplicate rows: 0


,Missing Values
customer_id,0
recieved_datatime,0
platform,0
type,0



Server Logs
Rows: 67249
Duplicate rows: 0


,Missing Values
datetime,0
session_id,0
ip,0
login,0
user_agent,0



Survey
Rows: 221
Duplicate rows: 0


,Missing Values
email,0
reported_income,0
nps_recommend,0
satisfaction_score,0
online_comparison_freq,0


In [8]:

def extract_table(connection, table_name):
    query = f'SELECT * FROM "{table_name}"'
    return pd.read_sql_query(query, connection)

database_tables = {}

for table_name in table_names:
    database_tables[table_name] = extract_table(
        connection,
        table_name
    )
    print(table_name, database_tables[table_name].shape)


customer (221, 9)
payment (23188, 4)
product (194, 4)
product_category (26, 2)
product_review (10990, 5)
sale (23188, 6)
sale_product (59759, 4)
sqlite_sequence (9, 2)
subscription (221, 5)
subscription_type (3, 3)


In [9]:

for table_name, dataframe in database_tables.items():
    print("\nTable:", table_name)
    print("Columns:", dataframe.columns.tolist())
    display(dataframe.head(3))



Table: customer
Columns: ['id', 'name', 'email', 'password_hash', 'address_line_1', 'address_line_2', 'county', 'postcode', 'subscription_id']


,id,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1
1,2,Leyla Mammadova,Mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2
2,3,Sana Shah,Sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3



Table: payment
Columns: ['id', 'amount', 'payment_gateway', 'transaction_reference']


,id,amount,payment_gateway,transaction_reference
0,1,299,paypool,41f6e886f30a470e83c86a8b3fb0602f
1,2,299,cardpay,cbd0263851ac45d395b009c895f4b673
2,3,1199,directdebit,50b41d8af9584ff098c3d51caae933ce



Table: product
Columns: ['id', 'name', 'price', 'product_category_id']


,id,name,price,product_category_id
0,1,Premium 1 month subscription,1199,NaN
1,2,Basic 1 month subscription,799,NaN
2,3,SuperSaver 1 month subscription,299,NaN


... (10 outputs left out)


Table: subscription
Columns: ['id', 'start_date', 'end_date', 'auto_renew', 'subscription_type_id']


,id,start_date,end_date,auto_renew,subscription_type_id
0,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,3
1,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,3
2,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,1



Table: subscription_type
Columns: ['id', 'name', 'delivery_service']


,id,name,delivery_service
0,1,Premium,SignedFor24
1,2,Basic,Tracked48
2,3,SuperSaver,Economy72


In [10]:

subscription_tables = [
    name for name in database_tables
    if "subscription" in name.lower()
]

print("Subscription tables:", subscription_tables)

for name in subscription_tables:
    display(database_tables[name].head())
    print(database_tables[name].dtypes)


Subscription tables: ['subscription', 'subscription_type']


,id,start_date,end_date,auto_renew,subscription_type_id
0,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,3
1,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,3
2,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,1
3,4,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,1
4,5,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,0,3


id                       int64
start_date              object
end_date                object
auto_renew               int64
subscription_type_id     int64
dtype: object


,id,name,delivery_service
0,1,Premium,SignedFor24
1,2,Basic,Tracked48
2,3,SuperSaver,Economy72


id                   int64
name                object
delivery_service    object
dtype: object


In [11]:

for table_name, dataframe in database_tables.items():
    relevant_columns = [
        column for column in dataframe.columns
        if any(
            keyword in column.lower()
            for keyword in [
                "customer",
                "subscription",
                "email",
                "sale",
                "order"
            ]
        )
    ]

    print(table_name, relevant_columns)


customer ['email', 'subscription_id']
payment []
product []
product_category []
product_review ['customer_id', 'sale_id']
sale ['order_date', 'customer_id']
sale_product ['sale_id']
sqlite_sequence []
subscription ['subscription_type_id']
subscription_type []


In [12]:

customer_df = database_tables["customer"].copy()
subscription_df = database_tables["subscription"].copy()
subscription_type_df = database_tables["subscription_type"].copy()
sales_df = database_tables["sale"].copy()

print("Customers:", customer_df.shape)
print("Subscriptions:", subscription_df.shape)
print("Subscription types:", subscription_type_df.shape)
print("Sales:", sales_df.shape)

display(customer_df.head())
display(sales_df.head())


Customers: (221, 9)
Subscriptions: (221, 5)
Subscription types: (3, 3)
Sales: (23188, 6)


,id,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1
1,2,Leyla Mammadova,Mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2
2,3,Sana Shah,Sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3
3,4,Omar Farooq,Omar.Farooq4@example.com,b21911abfc36881eea1278dc19b3f6b1b652ec4c213c00...,22 High Street,Tonbridge,Kent,TN9 1LM,4
4,5,Alejandro Blanco,Alejandro357@somemail.co.uk,fadf0729d5b6518a26ba92b664122884d15c4f646eac03...,31 New Road,Radlett,Hertfordshire,WD7 8PQ,5


,id,order_date,expected_date,delivered_date,customer_id,payment_id
0,1,2026-01-01 00:00:00.000,None,None,1,1
1,2,2026-01-01 00:00:00.000,None,None,2,2
2,3,2026-01-01 00:00:00.000,None,None,3,3
3,4,2026-01-01 00:00:00.000,None,None,4,4
4,5,2026-01-01 00:00:00.000,None,None,5,5


In [13]:

def clean_customers(customers):
    df = customers.copy()
    df = df.drop_duplicates(subset=["id"])
    df["email"] = df["email"].astype("string").str.strip().str.lower()
    return df

def clean_subscriptions(subscriptions):
    df = subscriptions.copy()
    df = df.drop_duplicates(subset=["id"])
    df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
    df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce")
    df["auto_renew"] = pd.to_numeric(df["auto_renew"], errors="coerce")
    return df

customer_clean = clean_customers(customer_df)
subscription_clean = clean_subscriptions(subscription_df)

print("Customers:", customer_clean.shape)
print("Subscriptions:", subscription_clean.shape)


Customers: (221, 9)
Subscriptions: (221, 5)


In [14]:

def create_churn_dataset(customers, subscriptions, subscription_types):
    df = customers.merge(
        subscriptions,
        left_on="subscription_id",
        right_on="id",
        how="left",
        validate="many_to_one",
        suffixes=("_customer", "_subscription")
    )

    types = subscription_types[
        ["id", "name", "delivery_service"]
    ].rename(
        columns={
            "id": "subscription_type_id",
            "name": "subscription_type",
        }
    )

    df = df.merge(
        types,
        on="subscription_type_id",
        how="left",
        validate="many_to_one"
    )

    df["churn"] = np.where(
        df["auto_renew"].eq(0),
        1,
        np.where(df["auto_renew"].eq(1), 0, np.nan)
    )

    return df

churn_df = create_churn_dataset(
    customer_clean,
    subscription_clean,
    subscription_type_df
)

print("Customer records:", len(churn_df))
print(churn_df["churn"].value_counts(dropna=False))
display(churn_df.head())


Customer records: 221
churn
0.0    135
1.0     86
Name: count, dtype: int64


,id_customer,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id,id_subscription,start_date,end_date,auto_renew,subscription_type_id,subscription_type,delivery_service,churn
0,1,Anahit Sargsyan,sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1,1,2026-01-01,2026-01-31,0,3,SuperSaver,Economy72,1.0
1,2,Leyla Mammadova,mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2,2,2026-01-01,2026-11-27,1,3,SuperSaver,Economy72,0.0
2,3,Sana Shah,sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3,3,2026-01-01,2026-11-27,1,1,Premium,SignedFor24,0.0
3,4,Omar Farooq,omar.farooq4@example.com,b21911abfc36881eea1278dc19b3f6b1b652ec4c213c00...,22 High Street,Tonbridge,Kent,TN9 1LM,4,4,2026-01-01,2026-11-27,1,1,Premium,SignedFor24,0.0
4,5,Alejandro Blanco,alejandro357@somemail.co.uk,fadf0729d5b6518a26ba92b664122884d15c4f646eac03...,31 New Road,Radlett,Hertfordshire,WD7 8PQ,5,5,2026-01-01,2026-05-01,0,3,SuperSaver,Economy72,1.0


In [15]:

total_customers = len(churn_df)
valid_customers = churn_df["churn"].notna().sum()
churned_customers = int(churn_df["churn"].eq(1).sum())
active_renewal_customers = int(churn_df["churn"].eq(0).sum())

churn_rate = churned_customers / valid_customers * 100

print("Total customers:", total_customers)
print("Auto-renewal disabled:", churned_customers)
print("Auto-renewal enabled:", active_renewal_customers)
print(f"Churn rate: {churn_rate:.2f}%")


Total customers: 221
Auto-renewal disabled: 86
Auto-renewal enabled: 135
Churn rate: 38.91%


In [16]:

def clean_crm(crm):
    df = crm.copy()
    df = df.drop_duplicates()
    df["customer_id"] = pd.to_numeric(
        df["customer_id"], errors="coerce"
    )
    df["recieved_datatime"] = pd.to_datetime(
        df["recieved_datatime"], errors="coerce"
    )
    df["type"] = df["type"].astype("string").str.strip().str.lower()
    return df

print(crm_df.columns.tolist())


['customer_id', 'recieved_datatime', 'platform', 'type']


In [17]:

crm_clean = crm_df.copy()
crm_clean["type"] = crm_clean["type"].astype("string").str.strip().str.lower()

crm_features = crm_clean.groupby("customer_id").agg(
    total_interactions=("type", "size"),
    complaint_count=("type", lambda x: (x == "complaint").sum()),
    query_count=("type", lambda x: (x == "query").sum())
).reset_index()

print("CRM customer records:", len(crm_features))
display(crm_features.head())


CRM customer records: 200


,customer_id,total_interactions,complaint_count,query_count
0,1,3,2,1
1,2,45,21,24
2,3,167,90,77
3,4,281,132,149
4,5,5,1,4


In [18]:

def aggregate_logins(server_logs):
    df = server_logs.copy()
    df["login"] = df["login"].astype("string").str.strip().str.lower()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

    features = df.groupby("login").agg(
        total_logins=("session_id", "count"),
        unique_sessions=("session_id", "nunique"),
        first_login=("datetime", "min"),
        last_login=("datetime", "max")
    ).reset_index()

    return features.rename(columns={"login": "email"})

login_features = aggregate_logins(server_df)

print("Customers with login activity:", len(login_features))
display(login_features.head())


Customers with login activity: 221


,email,total_logins,unique_sessions,first_login,last_login
0,a-davis569@somemail.co.uk,18,18,2026-10-31 06:34:14.264,2026-11-07 18:11:22.440
1,a.adams90@somemail.co.uk,562,562,2026-01-14 19:35:32.327,2026-11-07 17:54:02.439
2,a.ionescu6@somemail.co.uk,614,614,2026-02-09 21:41:46.956,2026-11-09 19:47:20.493
3,a.kamara264@example.com,123,123,2026-03-06 21:48:42.556,2026-04-16 18:58:02.529
4,a.nair119@somemail.co.uk,74,74,2026-07-26 20:22:23.954,2026-11-09 18:50:00.490


In [19]:

def clean_survey(survey):
    df = survey.copy()
    df["email"] = df["email"].astype("string").str.strip().str.lower()

    numeric_columns = [
        "nps_recommend",
        "satisfaction_score",
        "online_comparison_freq"
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df = df.drop_duplicates(subset=["email"])

    return df

survey_clean = clean_survey(survey_df)

print("Survey records:", len(survey_clean))
display(survey_clean.head())


Survey records: 221


,email,reported_income,nps_recommend,satisfaction_score,online_comparison_freq
0,sargsyan701@somemail.co.uk,26k-30k,8,6,5
1,mammadova282@example.com,34k-38k,5,8,9
2,sana858@notrealemail.com,26k-30k,9,5,5
3,omar.farooq4@example.com,198k-200k+,9,3,5
4,alejandro357@somemail.co.uk,42k-46k,0,8,10


In [20]:

customer_id_column = "id_customer" if "id_customer" in churn_df.columns else "id"

if "id_customer" not in churn_df.columns and "id" not in churn_df.columns:
    raise KeyError("Customer ID column not found")

final_df = churn_df.merge(
    crm_features,
    left_on=customer_id_column,
    right_on="customer_id",
    how="left",
    validate="one_to_one"
)

final_df = final_df.merge(
    login_features,
    on="email",
    how="left",
    validate="many_to_one"
)

final_df = final_df.merge(
    survey_clean,
    on="email",
    how="left",
    validate="many_to_one"
)

count_columns = [
    "total_interactions",
    "complaint_count",
    "query_count",
    "total_logins",
    "unique_sessions"
]

final_df[count_columns] = final_df[count_columns].fillna(0)

print("Final dataset shape:", final_df.shape)
display(final_df.head())


Final dataset shape: (221, 29)


,id_customer,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id,id_subscription,...,complaint_count,query_count,total_logins,unique_sessions,first_login,last_login,reported_income,nps_recommend,satisfaction_score,online_comparison_freq
0,1,Anahit Sargsyan,sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1,1,...,2.0,1.0,48,48,2026-01-01 17:28:25.008,2026-01-11 15:41:35.247,26k-30k,8,6,5
1,2,Leyla Mammadova,mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2,2,...,21.0,24.0,1085,1085,2026-01-01 16:48:25.008,2026-11-09 21:35:20.498,34k-38k,5,8,9
2,3,Sana Shah,sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3,3,...,90.0,77.0,624,624,2026-01-01 17:40:25.009,2026-11-09 20:43:20.496,26k-30k,9,5,5
3,4,Omar Farooq,omar.farooq4@example.com,b21911abfc36881eea1278dc19b3f6b1b652ec4c213c00...,22 High Street,Tonbridge,Kent,TN9 1LM,4,4,...,132.0,149.0,1540,1540,2026-01-01 14:35:05.006,2026-11-09 21:56:40.499,198k-200k+,9,3,5
4,5,Alejandro Blanco,alejandro357@somemail.co.uk,fadf0729d5b6518a26ba92b664122884d15c4f646eac03...,31 New Road,Radlett,Hertfordshire,WD7 8PQ,5,5,...,1.0,4.0,388,388,2026-01-01 17:51:05.010,2026-04-13 20:45:25.463,42k-46k,0,8,10


In [21]:

print(churn_df.columns.tolist())


['id_customer', 'name', 'email', 'password_hash', 'address_line_1', 'address_line_2', 'county', 'postcode', 'subscription_id', 'id_subscription', 'start_date', 'end_date', 'auto_renew', 'subscription_type_id', 'subscription_type', 'delivery_service', 'churn']


In [22]:

sales_clean = sales_df.copy()

print("Sales records:", len(sales_clean))
print("Sales columns:", sales_clean.columns.tolist())

display(sales_clean.head())


Sales records: 23188
Sales columns: ['id', 'order_date', 'expected_date', 'delivered_date', 'customer_id', 'payment_id']


,id,order_date,expected_date,delivered_date,customer_id,payment_id
0,1,2026-01-01 00:00:00.000,None,None,1,1
1,2,2026-01-01 00:00:00.000,None,None,2,2
2,3,2026-01-01 00:00:00.000,None,None,3,3
3,4,2026-01-01 00:00:00.000,None,None,4,4
4,5,2026-01-01 00:00:00.000,None,None,5,5


In [23]:

def create_sales_features(sales):
    df = sales.copy()

    df["order_date"] = pd.to_datetime(
        df["order_date"],
        errors="coerce"
    )

    features = df.groupby("customer_id").agg(
        total_orders=("id", "nunique"),
        first_order=("order_date", "min"),
        last_order=("order_date", "max")
    ).reset_index()

    return features

sales_features = create_sales_features(sales_clean)

print("Sales feature records:", len(sales_features))
display(sales_features.head())


Sales feature records: 221


,customer_id,total_orders,first_order,last_order
0,1,20,2026-01-01,2026-01-10 20:43:36.234
1,2,253,2026-01-01,2026-11-09 21:35:20.498
2,3,304,2026-01-01,2026-11-07 21:31:22.450
3,4,1055,2026-01-01,2026-11-09 21:56:40.499
4,5,44,2026-01-01,2026-04-12 19:52:46.436


In [24]:

final_df = final_df.merge(
    sales_features,
    left_on="id_customer",
    right_on="customer_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_sales")
)

final_df["total_orders"] = final_df["total_orders"].fillna(0)

print("Dataset shape:", final_df.shape)
display(final_df.head())


Dataset shape: (221, 33)


,id_customer,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id,id_subscription,...,first_login,last_login,reported_income,nps_recommend,satisfaction_score,online_comparison_freq,customer_id_sales,total_orders,first_order,last_order
0,1,Anahit Sargsyan,sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1,1,...,2026-01-01 17:28:25.008,2026-01-11 15:41:35.247,26k-30k,8,6,5,1,20,2026-01-01,2026-01-10 20:43:36.234
1,2,Leyla Mammadova,mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2,2,...,2026-01-01 16:48:25.008,2026-11-09 21:35:20.498,34k-38k,5,8,9,2,253,2026-01-01,2026-11-09 21:35:20.498
2,3,Sana Shah,sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3,3,...,2026-01-01 17:40:25.009,2026-11-09 20:43:20.496,26k-30k,9,5,5,3,304,2026-01-01,2026-11-07 21:31:22.450
3,4,Omar Farooq,omar.farooq4@example.com,b21911abfc36881eea1278dc19b3f6b1b652ec4c213c00...,22 High Street,Tonbridge,Kent,TN9 1LM,4,4,...,2026-01-01 14:35:05.006,2026-11-09 21:56:40.499,198k-200k+,9,3,5,4,1055,2026-01-01,2026-11-09 21:56:40.499
4,5,Alejandro Blanco,alejandro357@somemail.co.uk,fadf0729d5b6518a26ba92b664122884d15c4f646eac03...,31 New Road,Radlett,Hertfordshire,WD7 8PQ,5,5,...,2026-01-01 17:51:05.010,2026-04-13 20:45:25.463,42k-46k,0,8,10,5,44,2026-01-01,2026-04-12 19:52:46.436


In [25]:

sensitive_columns = [
    "password_hash",
    "name",
    "email",
    "address_line_1",
    "address_line_2",
    "county",
    "postcode",
    "ip",
    "session_id"
]

analysis_df = final_df.drop(
    columns=sensitive_columns,
    errors="ignore"
).copy()

print("Analysis dataset:", analysis_df.shape)
display(analysis_df.head())


Analysis dataset: (221, 26)


,id_customer,subscription_id,id_subscription,start_date,end_date,auto_renew,subscription_type_id,subscription_type,delivery_service,churn,...,first_login,last_login,reported_income,nps_recommend,satisfaction_score,online_comparison_freq,customer_id_sales,total_orders,first_order,last_order
0,1,1,1,2026-01-01,2026-01-31,0,3,SuperSaver,Economy72,1.0,...,2026-01-01 17:28:25.008,2026-01-11 15:41:35.247,26k-30k,8,6,5,1,20,2026-01-01,2026-01-10 20:43:36.234
1,2,2,2,2026-01-01,2026-11-27,1,3,SuperSaver,Economy72,0.0,...,2026-01-01 16:48:25.008,2026-11-09 21:35:20.498,34k-38k,5,8,9,2,253,2026-01-01,2026-11-09 21:35:20.498
2,3,3,3,2026-01-01,2026-11-27,1,1,Premium,SignedFor24,0.0,...,2026-01-01 17:40:25.009,2026-11-09 20:43:20.496,26k-30k,9,5,5,3,304,2026-01-01,2026-11-07 21:31:22.450
3,4,4,4,2026-01-01,2026-11-27,1,1,Premium,SignedFor24,0.0,...,2026-01-01 14:35:05.006,2026-11-09 21:56:40.499,198k-200k+,9,3,5,4,1055,2026-01-01,2026-11-09 21:56:40.499
4,5,5,5,2026-01-01,2026-05-01,0,3,SuperSaver,Economy72,1.0,...,2026-01-01 17:51:05.010,2026-04-13 20:45:25.463,42k-46k,0,8,10,5,44,2026-01-01,2026-04-12 19:52:46.436


In [26]:

print("Total rows:", len(analysis_df))
print("Unique customers:", analysis_df["id_customer"].nunique())
print("Duplicate customers:", analysis_df["id_customer"].duplicated().sum())
print("Missing churn values:", analysis_df["churn"].isna().sum())

display(
    analysis_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)


Total rows: 221
Unique customers: 221
Duplicate customers: 0
Missing churn values: 0


,Missing Values
customer_id,21
id_customer,0
total_logins,0
first_order,0
total_orders,0
customer_id_sales,0
online_comparison_freq,0
satisfaction_score,0
nps_recommend,0
reported_income,0


In [27]:

assert len(analysis_df) == len(customer_clean)
assert analysis_df["id_customer"].is_unique
assert analysis_df["churn"].notna().all()
assert analysis_df["churn"].isin([0, 1]).all()
assert analysis_df["total_orders"].ge(0).all()

print("All validation checks passed")


All validation checks passed


In [28]:

churn_summary = analysis_df.groupby("churn").agg(
    customers=("id_customer", "count"),
    average_orders=("total_orders", "mean"),
    average_complaints=("complaint_count", "mean"),
    average_logins=("total_logins", "mean"),
    average_satisfaction=("satisfaction_score", "mean")
).reset_index()

churn_summary["churn_status"] = churn_summary["churn"].map({
    0: "Auto-renewal enabled",
    1: "Auto-renewal disabled"
})

display(churn_summary)


,churn,customers,average_orders,average_complaints,average_logins,average_satisfaction,churn_status
0,0.0,135,121.859259,20.681481,350.748148,4.622222,Auto-renewal enabled
1,1.0,86,78.337209,12.151163,231.372093,4.976744,Auto-renewal disabled


In [29]:

output_folder = Path.cwd() / "output"
output_folder.mkdir(exist_ok=True)

output_path = output_folder / "PerfectPetClub_Churn.parquet"

analysis_df.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print("Parquet file created:", output_path)
print("File size:", output_path.stat().st_size, "bytes")


Parquet file created: /Users/akashchowdary/PerfectPetClub_Project/output/PerfectPetClub_Churn.parquet
File size: 38082 bytes


In [30]:

verified_df = pd.read_parquet(output_path)

assert len(verified_df) == len(analysis_df)
assert verified_df["id_customer"].is_unique

print("Parquet verification successful")
print("Rows:", verified_df.shape[0])
print("Columns:", verified_df.shape[1])

display(verified_df.head())


Parquet verification successful
Rows: 221
Columns: 26


,id_customer,subscription_id,id_subscription,start_date,end_date,auto_renew,subscription_type_id,subscription_type,delivery_service,churn,...,first_login,last_login,reported_income,nps_recommend,satisfaction_score,online_comparison_freq,customer_id_sales,total_orders,first_order,last_order
0,1,1,1,2026-01-01,2026-01-31,0,3,SuperSaver,Economy72,1.0,...,2026-01-01 17:28:25.008,2026-01-11 15:41:35.247,26k-30k,8,6,5,1,20,2026-01-01,2026-01-10 20:43:36.234
1,2,2,2,2026-01-01,2026-11-27,1,3,SuperSaver,Economy72,0.0,...,2026-01-01 16:48:25.008,2026-11-09 21:35:20.498,34k-38k,5,8,9,2,253,2026-01-01,2026-11-09 21:35:20.498
2,3,3,3,2026-01-01,2026-11-27,1,1,Premium,SignedFor24,0.0,...,2026-01-01 17:40:25.009,2026-11-09 20:43:20.496,26k-30k,9,5,5,3,304,2026-01-01,2026-11-07 21:31:22.450
3,4,4,4,2026-01-01,2026-11-27,1,1,Premium,SignedFor24,0.0,...,2026-01-01 14:35:05.006,2026-11-09 21:56:40.499,198k-200k+,9,3,5,4,1055,2026-01-01,2026-11-09 21:56:40.499
4,5,5,5,2026-01-01,2026-05-01,0,3,SuperSaver,Economy72,1.0,...,2026-01-01 17:51:05.010,2026-04-13 20:45:25.463,42k-46k,0,8,10,5,44,2026-01-01,2026-04-12 19:52:46.436


In [31]:

connection.close()

print("SQLite connection closed")


SQLite connection closed
